# 🚀 GIAI ĐOẠN 5 — KIẾN TRÚC STAIR5-v2 / C-HET: CONFIDENCE-CALIBRATED EDGE TRUST
### 🏆 Kaggle ML Engineering Pipeline — STAIR with Behavioral Edge Calibration & Optional Audited Lorentz Contrastive Regularization

> **Đề tài:** Recommender Systems using Graph Representation: Multi-modal  
> **Khóa luận tốt nghiệp:** Khóa 2021–2025 — Khoa Công nghệ Thông tin, Trường Đại học Khoa học Tự nhiên, ĐHQG-HCM  
> **Sinh viên thực hiện:** Lê Hà Thanh Chương (23120195) & Bùi Trung Hiếu (23120257)  
> **Giảng viên hướng dẫn:** TS. Nguyễn Ngọc Thảo  
> **Kiến trúc:** **STAIR5-v2 / C-HET** (*Confidence-calibrated Edge Trust with optional Lorentz Hidden Contrastive regularization*)  
> **Tài liệu phương pháp luận & Thiết kế:** `docs/giai_doan_5/STAIR5_v2_Report.md` & `docs/giai_doan_5/STAIR5_v2_Implementation.md`  
> **Mã nguồn lõi:** `models/stair5_v2.py`, `models/stair5_v2_graph.py`, `models/stair5_v2_geometry.py`, `models/stair5_v2_objectives.py`, `optimizers/stair5_v2_smoother.py`, `main_stair5_v2.py`  
> **Tập dữ liệu mục tiêu:** **Amazon Sports**, **Amazon Baby**, **Amazon Electronics** (3 tập chuẩn thương mại điện tử)  
> **Chiến lược thực thi:** **3-Tier Fail-Fast Pipeline** (`Pha A: Sports [Pilot] ➔ Pha B: Baby [Confirmatory] ➔ Pha C: Electronics [Scale-up]`)  
> **Đo lường VRAM:** Paper Standard (`torch.cuda.max_memory_allocated()`, trích xuất trực tiếp từ `training_telemetry.jsonl` và host NVML)

---

## 📑 1. BẢN THIẾT KẾ ĐIỀU CHỈNH CHUẨN TẮC: STAIR5-v2 / C-HET

| Trụ Cột Thành Phần | Cơ Chế Toán Học & Kỹ Thuật Thực Thi | Lợi Ích Vận Hành & Khắc Phục Lỗ Hổng Kỹ Thuật |
|:---|:---|:---|
| **1. Bảo Tồn 100% Backbone STAIR** | MI SVD Whitening, Tích chập phổ FSC $\beta = 1 - \beta_3$, toán tử BSC Smoother trong AdamWSEvo, hàm mất mát BPR, suy luận tích vô hướng Euclidean 100%. | Giữ nguyên vẹn độ trễ suy luận $O(1)$ (Zero Inference Overhead); bảo đảm baseline recovery chính xác khi $a=0$ hoặc $\eta=0$. |
| **2. Candidate-Restricted Co-occurrence** | Đếm $n_{ij} = \sum_u R_{ui} R_{uj}$ chỉ trên tập cạnh candidate $C$ từ baseline kNN. Dùng sorted user lists và binary search. | Tuyệt đối không dựng ma trận dày $I \times I$ hoặc tính $R^\top R$; tiết kiệm bộ nhớ trên tập khổng lồ Electronics. |
| **3. Ochiai × Support Shrinkage** | $b_{ij} = n_{ij} / \sqrt{d_i d_j}$, $\rho_{ij} = n_{ij} / (n_{ij} + t)$, score tin cậy $q_{ij} = \rho_{ij} b_{ij} \in [0, 1]$. | Giảm can thiệp trên cạnh ít bằng chứng ($t=5$); loại trừ nhiễu đồng tương tác ngẫu nhiên của các item ít tương tác. |
| **4. Raw Edge Boost & Operator Blend** | $W^+_{ij} = W^0_{ij}(1 + a q_{ij})$, $S^+ = (D^+)^{-1/2} W^+ (D^+)^{-1/2}$, $S_\eta = (1 - \eta)S_0 + \eta S^+$. | Hiệu chỉnh trọng số cạnh đa phương thái dựa trên hành vi thực tế; giữ chặn phổ $\|S_\eta\|_2 \le 1$. |
| **5. Sparse Matrix-Free BSC Smoother** | $P_j(S_\eta) G_j = \frac{1 - b_j}{1 - b_j^{L+1}} \sum_{\ell=0}^L b_j^\ell S_\eta^\ell G_j$. Callback giải quyết buffer động. | Không lưu ma trận $S^2$; thực thi qua phép nhân thưa ma trận CSR $\times$ vector dày cực nhanh trên GPU. |
| **6. Checkpointed Audited LHC (Tùy chọn)** | Multi-positive CE đối xứng $U_0 \leftrightarrow I_1$, $I_0 \leftrightarrow U_1$, chia block query 256 với activation checkpointing không giữ activation map. | Tiêu thụ VRAM tối thiểu; bảo đảm FP32 Lorentz geodesic Taylor bậc 4 triệt tiêu $100\%$ NaN. Nhánh chính ET tắt LHC ($\lambda=0$). |
| **7. Degree-Stratified Placebo Control** | Hoán vị $q_{ij}$ trong các bin bậc log2 endpoint để kiểm chứng xem lợi ích có thực sự đến từ cấu trúc đồ thị hay chỉ do phân phối trọng số. | Cung cấp đối chứng khoa học chặt chẽ, loại bỏ hoàn toàn ngụy biện mức độ phổ biến (popularity confounding). |

```
                ┌────────────────────────────────────────────────────────┐
                │             STAIR5-v2 / C-HET ARCHITECTURE             │
                └────────────────────────────────────────────────────────┘
                                             │
      ┌──────────────────────────────────────┴──────────────────────────────────────┐
      ▼                                                                             ▼
[CALIBRATED BSC OPERATOR]                                                 [RECOMMENDATION BACKBONE]
Modality Features ➔ Baseline kNN ➔ Raw W0, S0                              User-Item Graph ➔ FSC Convolution
Train Interactions ➔ Candidate Co-occurrence ➔ q_ij                        Euclidean Dot-Product Scorer ➔ BPR
      ↓                                                                             ↓
W+ = W0 ⊙ (1 + a q) ➔ S+ = (D+)^(-1/2) W+ (D+)^(-1/2)                      L_BPR Loss
      ↓                                                                             ↓
Sη = (1 - η) S0 + η S+                                                     Item Gradients Smoothed by Sη
(Materialized as Sparse CSR, zero dense memory)                                     ↓
                                                                  [Optional Audited Geodesic LHC]

## Cell 1 ⚙️ Thiết lập Môi trường, Dependencies & Đồng bộ Mã Nguồn STAIR5-v2
- Đồng bộ mã nguồn mới nhất từ GitHub repository `ThanhChuong12/STAIR-Enhanced` (branch `main`).
- Cài đặt các gói phụ thuộc chuẩn tắc tương thích với môi trường nghiệm thu: `freerec>=0.9.7`, `torchdata==0.8.0`, `torch-geometric`, `nvidia-ml-py`, `prettytable`, `matplotlib`, `pyyaml`, `scipy`, `pandas`.
- Khởi chạy tiến trình con xác thực (`Subprocess Environment Verification`) để kiểm tra toàn bộ imports và GPU runtime.

In [ ]:
# Cell 1: Môi trường, Dependencies & Đồng bộ STAIR-Enhanced (STAIR5-v2 C-HET)
import os, shutil, subprocess, sys, types, glob, torch
from pathlib import Path

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working') if os.path.exists('/kaggle/working') else None

# 1. Đồng bộ repository STAIR-Enhanced thông minh & an toàn
if os.path.exists(STAIR_DIR):
    print("Thư mục STAIR-Enhanced đã tồn tại. Đang kiểm tra đồng bộ mã nguồn...")
    try:
        subprocess.run(['git', '-C', STAIR_DIR, 'fetch', 'origin', 'main'], check=True, capture_output=True)
        chk = subprocess.run(['git', '-C', STAIR_DIR, 'ls-tree', 'origin/main', 'models/stair5_v2.py'], capture_output=True, text=True)
        if 'models/stair5_v2.py' in chk.stdout:
            subprocess.run(['git', '-C', STAIR_DIR, 'reset', '--hard', 'origin/main'], check=True)
            print("✅ Đã đồng bộ về commit mới nhất của origin/main (chứa STAIR5-v2 C-HET).")
        else:
            print("⚠️ Remote origin/main chưa có models/stair5_v2.py. Giữ nguyên mã nguồn hiện có trên đĩa.")
    except Exception as e:
        print(f"⚠️ Cảnh báo git fetch/check ({e}). Tiếp tục dùng mã nguồn hiện có trên đĩa.")

if not os.path.exists(STAIR_DIR) and os.path.exists('/kaggle/working'):
    print("Cloning STAIR-Enhanced repository (branch main)...")
    try:
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/ThanhChuong12/STAIR-Enhanced.git', STAIR_DIR
        ], check=True)
    except Exception as e:
        print(f"⚠️ Git clone gặp cảnh báo: {e}")

# Fallback: Tự động dò quét và sao chép mã nguồn từ Kaggle Input nếu chưa có models/stair5_v2.py
if not os.path.exists(os.path.join(STAIR_DIR, 'models', 'stair5_v2.py')):
    input_matches = glob.glob('/kaggle/input/**/models/stair5_v2.py', recursive=True)
    if input_matches:
        src_repo = os.path.dirname(os.path.dirname(input_matches[0]))
        print(f"📦 Phát hiện mã nguồn STAIR5-v2 từ Kaggle Input: {src_repo} -> Sao chép vào {STAIR_DIR}...")
        os.makedirs(STAIR_DIR, exist_ok=True)
        shutil.copytree(src_repo, STAIR_DIR, dirs_exist_ok=True)

active_dir = STAIR_DIR if os.path.exists(STAIR_DIR) else os.path.abspath('.')
for p in [active_dir, STAIR_DIR, '/kaggle/working', '.']:
    if p and os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

if os.path.exists(STAIR_DIR):
    os.chdir(STAIR_DIR)

# Thiết lập biến môi trường chuẩn deterministic và PYTHONPATH cho subprocess
sub_env = os.environ.copy()
sub_env['PYTHONPATH'] = f"{active_dir}:{os.environ.get('PYTHONPATH', '')}"
sub_env['PYTHONWARNINGS'] = 'ignore'
sub_env.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['PYTHONPATH'] = sub_env['PYTHONPATH']
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# Xóa cache module để kernel luôn nạp phiên bản v2 mới nhất từ đĩa
for mod_name in list(sys.modules.keys()):
    if any(k in mod_name for k in ['stair5_v2', 'models.stair5_v2', 'optimizers.stair5_v2_smoother']):
        sys.modules.pop(mod_name, None)

# 2. Cài đặt các gói phụ thuộc chuẩn tắc tương thích với môi trường nghiệm thu
print("Cài đặt dependencies tương thích (freerec, torchdata, torch-geometric, pyyaml, prettytable)...")
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'pynvml', 'torchdata'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchdata==0.8.0'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'freerec>=0.9.7', 'torch-geometric', 'nvidia-ml-py', 'prettytable', 'matplotlib', 'pyyaml', 'scipy', 'pandas'
], check=False)

# 3. Patch tương thích nội bộ PyTorch Dynamo & TorchData DataPipe Registration
try:
    import models.freerec_compat
except Exception:
    try:
        import freerec_compat
    except Exception:
        pass

# 4. Xác minh môi trường trong tiến trình con (Subprocess Verification)
verify_script = (
    "import sys; sys.path.insert(0, '" + active_dir + "'); "
    "import torch; import models.freerec_compat; import freerec, yaml; "
    "from models.stair5_v2 import STAIR5_v2_Model; "
    "print(f'[Subprocess Verified] Python={sys.version.split()[0]}, PyTorch={torch.__version__}, FreeRec={freerec.__version__}')"
)
verify_cmd = [sys.executable, '-W', 'ignore', '-c', verify_script]
res = subprocess.run(verify_cmd, capture_output=True, text=True, cwd=active_dir, env=sub_env)
if res.returncode != 0:
    print("❌ Lỗi khi xác minh môi trường tiến trình con:")
    print("STDOUT:", res.stdout)
    print("STDERR:", res.stderr)
    m_path = os.path.join(active_dir, 'models', 'stair5_v2.py')
    print(f"Kiểm tra file models/stair5_v2.py tồn tại: {os.path.exists(m_path)} (Đường dẫn: {m_path})")
    raise RuntimeError(f"Subprocess verification failed with exit code {res.returncode}:\n{res.stderr}")

import freerec
print('=' * 80)
print('THÔNG TIN MÔI TRƯỜNG THỰC THI (KAGGLE ML ENGINE — STAIR5-v2 C-HET):')
print(f'  * {res.stdout.strip()}')
print(f'  * CUDA Available     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  * GPU Model          : {torch.cuda.get_device_name(0)}')
    print(f'  * Total VRAM         : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
print(f'  * Working Directory  : {os.getcwd()}')

m_ok    = os.path.exists(os.path.join(active_dir, 'models', 'stair5_v2.py'))
g_ok    = os.path.exists(os.path.join(active_dir, 'models', 'stair5_v2_graph.py'))
geo_ok  = os.path.exists(os.path.join(active_dir, 'models', 'stair5_v2_geometry.py'))
obj_ok  = os.path.exists(os.path.join(active_dir, 'models', 'stair5_v2_objectives.py'))
sm_ok   = os.path.exists(os.path.join(active_dir, 'optimizers', 'stair5_v2_smoother.py'))
main_ok = os.path.exists(os.path.join(active_dir, 'main_stair5_v2.py'))
cfg_ok  = os.path.exists(os.path.join(active_dir, 'configs', 'Amazon2014Sports_STAIR5_v2.yaml'))

print(f'  * models/stair5_v2.py            : {"✅ SẴN SÀNG" if m_ok else "❌ CHƯA CÓ"}')
print(f'  * models/stair5_v2_graph.py      : {"✅ SẴN SÀNG" if g_ok else "❌ CHƯA CÓ"}')
print(f'  * models/stair5_v2_geometry.py   : {"✅ SẴN SÀNG" if geo_ok else "❌ CHƯA CÓ"}')
print(f'  * models/stair5_v2_objectives.py : {"✅ SẴN SÀNG" if obj_ok else "❌ CHƯA CÓ"}')
print(f'  * optimizers/stair5_v2_smoother  : {"✅ SẴN SÀNG" if sm_ok else "❌ CHƯA CÓ"}')
print(f'  * main_stair5_v2.py              : {"✅ SẴN SÀNG" if main_ok else "❌ CHƯA CÓ"}')
print(f'  * configs/*_STAIR5_v2.yaml       : {"✅ SẴN SÀNG" if cfg_ok else "❌ CHƯA CÓ"}')
print('=' * 80)

## Cell 2 📂 Chuẩn bị Dữ liệu từ Kaggle Input (Multi-Bridge sang /kaggle/data & Processed)
- Tự động dò quét toàn bộ kho dữ liệu Kaggle Input (`/kaggle/input`) để tìm kiếm 3 tập chuẩn: **Amazon Sports**, **Amazon Baby**, **Amazon Electronics**.
- Thiết lập cơ chế Symlink/Hardlink liên kết đa hướng sang `/kaggle/data`, `/kaggle/data/Processed`, `STAIR-Enhanced/data` để FreeRec luôn tìm thấy dữ liệu ở mọi vị trí.

In [ ]:
# Cell 2: Chuẩn bị dữ liệu từ Kaggle Input sang /kaggle/data & Processed
import os, shutil, glob

DATA_ROOT = '/kaggle/data'
PROCESSED_ROOT = os.path.join(DATA_ROOT, 'Processed')
LOCAL_DATA = '/kaggle/working/STAIR-Enhanced/data'
LOCAL_PROCESSED = os.path.join(LOCAL_DATA, 'Processed')

for d in [DATA_ROOT, PROCESSED_ROOT, LOCAL_DATA, LOCAL_PROCESSED]:
    os.makedirs(d, exist_ok=True)

TARGET_DATASETS = {
    'sports':      ('Amazon2014Sports_550_MMRec', ['sport', 'sports', 'amazon2014sports']),
    'baby':        ('Amazon2014Baby_550_MMRec', ['baby', 'amazon2014baby']),
    'electronics': ('Amazon2014Electronics_550_MMRec', ['electronic', 'electronics', 'amazon2014electronics']),
}

REQUIRED_EXTENSIONS = ('.npy', '.pkl', '.txt', '.inter', '.item', '.pt', '.csv', '.yaml')

def bridge_directories(src_dir, target_folder):
    '''Đồng bộ dữ liệu sang toàn bộ các vị trí FreeRec có thể tìm kiếm'''
    destinations = [
        os.path.join(DATA_ROOT, target_folder),
        os.path.join(PROCESSED_ROOT, target_folder),
        os.path.join(LOCAL_DATA, target_folder),
        os.path.join(LOCAL_PROCESSED, target_folder),
    ]
    for dst in destinations:
        if os.path.abspath(src_dir) == os.path.abspath(dst):
            continue
        os.makedirs(dst, exist_ok=True)
        for item in os.listdir(src_dir):
            s_item = os.path.join(src_dir, item)
            d_item = os.path.join(dst, item)
            if os.path.isdir(s_item):
                if not os.path.exists(d_item):
                    try:
                        os.symlink(s_item, d_item)
                    except Exception:
                        shutil.copytree(s_item, d_item, dirs_exist_ok=True)
            else:
                if not os.path.exists(d_item) or os.path.getsize(d_item) == 0:
                    try:
                        os.symlink(s_item, d_item)
                    except Exception:
                        shutil.copy2(s_item, d_item)

print("=" * 80)
print("TIẾN TRÌNH DÒ QUÉT & LIÊN KẾT DỮ LIỆU TỰ ĐỘNG CHO STAIR5-v2 C-HET:")
prepared_data = set()
search_bases = ['/kaggle/input', '.', '..', '/kaggle/working']

for key, (folder_name, aliases) in TARGET_DATASETS.items():
    found_src = None
    for base in search_bases:
        if not os.path.exists(base):
            continue
        for root, dirs, files in os.walk(base):
            bname = os.path.basename(root).lower()
            if bname == folder_name.lower() or any(alias in bname for alias in aliases):
                has_req = any(f.endswith(REQUIRED_EXTENSIONS) for f in files)
                if has_req:
                    found_src = root
                    break
        if found_src:
            break

    if found_src:
        bridge_directories(found_src, folder_name)
        prepared_data.add(key)
        item_count = len(os.listdir(found_src))
        print(f"  ✅ [SẴN SÀNG] {key.upper():12s} -> Nguồn: {found_src} ({item_count} files)")
    else:
        print(f"  ⚠️ [CHƯA THẤY] {key.upper():12s} -> Sẽ nạp tự động qua kịch bản hoặc config.")
print("=" * 80)

## Cell 3 🧪 Kiểm tra Toàn Diện Hệ Thống STAIR5-v2 (Bộ 22 Kiểm Thử Toán Học & Parity)
Chạy bộ kiểm thử độc lập bao gồm 22 bài kiểm tra nghiêm ngặt:
1. **Graph Calibration Tests (`tests/test_stair5_v2_graph.py` - 9 tests):**
   - Phục hồi chính xác baseline: $a=0$ hoặc $\eta=0$ trả về đúng đối tượng $S_0$ gốc.
   - Boost hằng số: triệt tiêu chính xác qua chuẩn hóa đối xứng.
   - Thống kê ứng viên: deduplicate tương tác nhị phân, chỉ tương tác train mới ảnh hưởng tới $q$.
   - Tính đối xứng, chặn phổ $\|S_\eta\|_2 \le 1$, và kiểm định hoán vị placebo theo strata bậc.
   - Invalidation cache khi thay đổi split hoặc siêu tham số.
2. **Audited Objectives Tests (`tests/test_stair5_v2_objectives.py` - 7 tests):**
   - Không gian Lorentz và khoảng cách tự thân bằng 0; Taylor series mượt mà loại bỏ NaN.
   - Chặn bán kính hữu hạn tại gốc và biên.
   - Khử shortcut self-return có mask bối cảnh rỗng.
   - Multi-positive CE khớp chính xác log-softmax với đối xứng 2 chiều.
   - Block checkpointing khớp loss và gradient của toàn bộ ma trận trong sai số FP32.
3. **Pipeline & Integration Tests (`tests/test_stair5_v2_pipeline.py` - 6 tests):**
   - Đối chiếu nguồn gốc: $B_0$ khớp 100% từng bước AdamWSEvo, encode, ranking và seen masking với `main.py`.
   - Phân lập nhóm tối ưu: Item có Smoother, User không có.
   - Khởi tạo và đóng băng các buffers thang đo.
   - Roundtrip checkpoint: phục hồi mô hình, optimizer moments và tiếp tục bước tối ưu tiếp theo.

In [ ]:
# Cell 3: Kiểm tra Hệ Thống STAIR5-v2 & Chạy Bộ 22 Unit Tests Toán Học
import os, sys, subprocess

for p in ['/kaggle/working/STAIR-Enhanced', os.path.abspath('.'), '.', '/kaggle/working']:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

print("=" * 80)
print("🚀 CHẠY BỘ KIỂM THỬ ĐỘC LẬP TOÀN DIỆN CHO STAIR5-v2 C-HET (22 TESTS)...")
print("=" * 80)

active_dir = '/kaggle/working/STAIR-Enhanced' if os.path.exists('/kaggle/working/STAIR-Enhanced') else os.path.abspath('.')

sub_env = os.environ.copy()
sub_env['PYTHONPATH'] = f"{active_dir}:{sub_env.get('PYTHONPATH', '')}"
sub_env['PYTHONWARNINGS'] = 'ignore'

test_cmd = [
    sys.executable, '-W', 'ignore', '-m', 'unittest', 'discover',
    '-s', os.path.join(active_dir, 'tests'),
    '-p', 'test_stair5_v2_*.py',
    '-v'
]

res = subprocess.run(test_cmd, capture_output=True, text=True, cwd=active_dir, env=sub_env)
print(res.stdout)
if res.stderr:
    print(res.stderr)

assert res.returncode == 0, f"Kiểm thử thất bại với exit code {res.returncode}!"
print("🎯 TẤT CẢ CÁC BÀI KIỂM TRA ĐÃ VƯỢT QUA 100% — KIẾN TRÚC STAIR5-v2 C-HET SẴN SÀNG HUẤN LUYỆN!")

## Cell 4 🛠️ Telemetry Engine: Training Runner, GPU VRAM Profiler & Visualization Utilities
- **Chuẩn đo lường VRAM:** Paper Standard (`torch.cuda.max_memory_allocated()`), trích xuất chính xác dung lượng bộ nhớ tensor thực tế qua các epoch từ `training_telemetry.jsonl` và theo dõi host GPU qua `pynvml`.
- **Phân tách Độc lập Loss:** Trích xuất **Pure BPR Loss** ($L_{BPR}$) và **Total Loss** ($L_{BPR} + \lambda(t) L_{LHC}$) trực tiếp từ telemetry động.
- **Trực quan hóa Đa chiều:** Biểu đồ 4 panel phản ánh: (a) Training Loss Trajectory, (b) Validation NDCG@20, (c) Validation Recall@20, và (d) Thông lượng huấn luyện (samples/s) kèm dung lượng VRAM thực tế (MiB).
- **Parser Chặt Chẽ:** Tách riêng `extract_best_validation(log_path)` và `extract_test_metrics(log_path)`. Chỉ nhận test metrics khi có khối `TEST @Epoch` chính thức từ việc nạp checkpoint tốt nhất theo chuẩn FreeRec.
- **Xử lý Thất bại Triệt để:** Ném `RuntimeError` dừng ngay tiến trình khi subprocess kết thúc với exit code khác 0, không che giấu lỗi.

In [ ]:
# Cell 4: Telemetry Engine — Training Runner, Hardware Profiler & Visualization
import os, sys, time, re, json, threading, subprocess
from pathlib import Path
import numpy as np

sys.stdout.reconfigure(encoding='utf-8') if hasattr(sys.stdout, 'reconfigure') else None

TRACKED_METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

BASELINE_REF = {
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V5_REF = {
    'sports':      {'Recall@10': 0.0753, 'Recall@20': 0.1113, 'NDCG@10': 0.0415, 'NDCG@20': 0.0508},
    'baby':        {'Recall@10': 0.0669, 'Recall@20': 0.1027, 'NDCG@10': 0.0362, 'NDCG@20': 0.0454},
    'electronics': {'Recall@10': 0.0451, 'Recall@20': 0.0678, 'NDCG@10': 0.0252, 'NDCG@20': 0.0311},
}

MHD_V3_REF = {
    'sports':      {'Recall@10': 0.0758, 'Recall@20': 0.1122, 'NDCG@10': 0.0418, 'NDCG@20': 0.0512},
    'baby':        {'Recall@10': 0.0678, 'Recall@20': 0.1030, 'NDCG@10': 0.0362, 'NDCG@20': 0.0452},
    'electronics': {'Recall@10': 0.0460, 'Recall@20': 0.0685, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
}

V1_H0_REF = {
    'sports':      {'Recall@10': 0.0747, 'Recall@20': 0.1133, 'NDCG@10': 0.0407, 'NDCG@20': 0.0506},
    'baby':        {'Recall@10': 0.0660, 'Recall@20': 0.1030, 'NDCG@10': 0.0352, 'NDCG@20': 0.0447},
    'electronics': {'Recall@10': 0.0435, 'Recall@20': 0.0666, 'NDCG@10': 0.0241, 'NDCG@20': 0.0301},
}

DATASET_PROFILES = {
    'sports': {
        'name':        'Amazon Sports',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '35,598 Users | 18,357 Items | 296K Interactions',
        'color':       '#ff7f0e',
        'approx_mins': 45.0,
    },
    'baby': {
        'name':        'Amazon Baby',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '19,445 Users | 7,050 Items | 160K Interactions',
        'color':       '#1f77b4',
        'approx_mins': 20.0,
    },
    'electronics': {
        'name':        'Amazon Electronics',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '192,403 Users | 63,001 Items | 1.69M Interactions',
        'color':       '#2ca02c',
        'approx_mins': 260.0,
    },
}

vram_profile = {}

def vram_monitor(key, stop_evt, interval=2.0):
    '''Background thread tracking host GPU allocated memory via NVML.'''
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        records = []
        while not stop_evt.is_set():
            mem = pynvml.nvmlDeviceGetMemoryInfo(h)
            records.append(mem.used / (1024**2))
            time.sleep(interval)
        pynvml.nvmlShutdown()
        vram_profile[key] = records
    except Exception:
        vram_profile[key] = []

def resolve_path(p):
    if os.path.exists(p):
        return p
    base = os.path.basename(p)
    candidates = [
        p,
        os.path.join('/kaggle/working/logs/stair5_v2', base),
        os.path.join('/kaggle/working/logs/GD5', base),
        os.path.join('logs/stair5_v2', base),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return p

def parse_telemetry_jsonl(artifact_dir):
    p = Path(artifact_dir) / "training_telemetry.jsonl"
    if not p.is_file():
        return []
    records = []
    with p.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    return records

def extract_best_validation(log_path):
    '''Extracts best validation epoch and validation metrics.'''
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    best_val_epoch = None
    best_val_metrics = {}
    best_ndcg20 = -1.0

    matches = re.finditer(r'VALID\s+@Epoch:\s*(\d+)(.*?)(?=(?:VALID|TEST|Load|TRAIN|\Z))', content, re.DOTALL)
    for m in matches:
        ep = int(m.group(1))
        block = m.group(2)
        ndcg_m = re.search(r'NDCG@20(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', block, re.IGNORECASE)
        if ndcg_m:
            val_ndcg20 = float(ndcg_m.group(1))
            if val_ndcg20 > best_ndcg20:
                best_ndcg20 = val_ndcg20
                best_val_epoch = ep
                cur_metrics = {'NDCG@20': val_ndcg20}
                for metric in ['Recall@10', 'Recall@20', 'NDCG@10']:
                    mm = re.search(rf'{metric}(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', block, re.IGNORECASE)
                    if mm:
                        cur_metrics[metric] = float(mm.group(1))
                best_val_metrics = cur_metrics
    return best_val_epoch, best_val_metrics

def extract_test_metrics(log_path):
    '''Strictly extracts test evaluation metrics ONLY after checkpoint load.'''
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    lines = content.splitlines()
    tail = '\n'.join(lines[-35:])
    if 'Traceback (most recent call last):' in tail:
        return None, {}

    best_match = re.search(r'\[Coach\]\s*>>>\s*Load best model\s*@Epoch:\s*(\d+)', content)
    if not best_match:
        return None, {}

    best_model_epoch = int(best_match.group(1))
    content_after_load = content[best_match.end():]

    test_match = re.search(r'TEST\s+@Epoch:\s*(\d+)(.*?)(?:=================|$)', content_after_load, re.DOTALL)
    if not test_match:
        return None, {}

    test_epoch = int(test_match.group(1))
    if test_epoch != best_model_epoch:
        return None, {}

    test_snippet = test_match.group(2)
    test_metrics = {}
    for metric in TRACKED_METRICS:
        m = re.search(rf'{metric}(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', test_snippet, re.IGNORECASE)
        if m:
            test_metrics[metric] = float(m.group(1))

    if len(test_metrics) < len(TRACKED_METRICS):
        return None, {}
    return test_epoch, test_metrics

def parse_training_losses(log_path, telemetry=None):
    '''Extracts pure BPR loss and total training loss trajectories.'''
    if telemetry and len(telemetry) > 0:
        bpr_losses = [(r['epoch'], float(r['bpr_loss'])) for r in telemetry if 'bpr_loss' in r]
        total_losses = [(r['epoch'], float(r['bpr_loss']) + float(r.get('lambda', 0.0)) * float(r.get('cl_loss', 0.0))) for r in telemetry if 'bpr_loss' in r]
        return bpr_losses, total_losses

    log_path = resolve_path(log_path)
    bpr_losses, total_losses = [], []
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        telemetry_matches = re.findall(
            r'\[Epoch\s+(\d+)\s+Telemetry\]\s+BPR:\s*([0-9.]+)\s+\|\s+LHC:\s*([0-9.]+)',
            content
        )
        if telemetry_matches:
            for ep_str, bpr_str, lhc_str in telemetry_matches:
                bpr_losses.append((int(ep_str), float(bpr_str)))
        loss_matches = re.findall(r'TRAIN @Epoch:\s*(\d+).*?LOSS\s+Avg:\s*([0-9.]+)', content, re.DOTALL)
        total_losses = [(int(ep), float(loss)) for ep, loss in loss_matches]
    return bpr_losses, total_losses

def parse_valid_metric(log_path, metric='NDCG@20'):
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    pattern = rf'VALID\s+@Epoch:\s*(\d+).*?{metric}\s+Avg:\s*([0-9.]+)'
    matches = re.findall(pattern, content, re.IGNORECASE)
    return [(int(ep), float(v)) for ep, v in matches]

def run_training_stair5_v2(key, config_yaml=None, data_root='/kaggle/data', log_path=None, artifact_dir=None, **kwargs):
    '''Executes STAIR5-v2 C-HET training with CLI argument overrides and strict error handling.'''
    cfg_dict = STAIR5_V2_CONFIGS.get(key, {}).copy()
    cfg_dict.update(kwargs)

    config_yaml = config_yaml or cfg_dict.get('yaml')
    log_path = log_path or cfg_dict.get('log')
    artifact_dir = artifact_dir or cfg_dict.get('artifact_dir')

    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    os.makedirs(artifact_dir, exist_ok=True)

    print('=' * 80)
    print(f'🚀 KHỞI ĐỘNG TIẾN TRÌNH HUẤN LUYỆN STAIR5-v2 / C-HET: {key.upper()}')
    print(f'  * Dataset Key         : {key}')
    print(f'  * YAML Configuration  : {config_yaml}')
    print(f'  * Log Path            : {log_path}')
    print(f'  * Artifact Directory  : {artifact_dir}')
    print(f'  * C-HET Arm           : {cfg_dict.get("v2_arm", "ET")}')
    print(f'  * Edge Strength (a)   : {cfg_dict.get("edge_strength", 0.25)}')
    print(f'  * Edge Mix (η)        : {cfg_dict.get("edge_mix", 0.25)}')
    print(f'  * Shrinkage (t)       : {cfg_dict.get("evidence_shrinkage", 5.0)}')
    print(f'  * KNN Device / Chunk  : {cfg_dict.get("knn_device", "auto")} / {cfg_dict.get("knn_chunk_size", 1024)}')
    print(f'  * Seed                : {cfg_dict.get("seed", 1)}')
    print('=' * 80)

    stop_evt = threading.Event()
    th = threading.Thread(target=vram_monitor, args=(key, stop_evt), daemon=True)
    th.start()

    t0 = time.time()
    runner_py = '/kaggle/working/STAIR-Enhanced/main_stair5_v2.py'
    if not os.path.exists(runner_py):
        runner_py = 'main_stair5_v2.py'

    if not os.path.exists(config_yaml):
        cand_y = os.path.join('configs', os.path.basename(config_yaml))
        if os.path.exists(cand_y):
            config_yaml = cand_y

    device_str = str(cfg_dict.get('device', '0' if torch.cuda.is_available() else 'cpu'))
    cmd = [
        sys.executable, '-u', runner_py,
        '--config', str(config_yaml),
        '--root', str(data_root),
        '--v2-arm', str(cfg_dict.get('v2_arm', 'ET')),
        '--seed', str(cfg_dict.get('seed', 1)),
        '--device', device_str,
        '--ranking', 'full',
        '--eval-freq', str(cfg_dict.get('eval_freq', 5)),
        '--knn-device', str(cfg_dict.get('knn_device', 'auto')),
        '--knn-chunk-size', str(cfg_dict.get('knn_chunk_size', 1024)),
        '--lhc-chunk-size', str(cfg_dict.get('lhc_chunk_size', 256)),
        '--edge-strength', str(cfg_dict.get('edge_strength', 0.25)),
        '--edge-mix', str(cfg_dict.get('edge_mix', 0.25)),
        '--evidence-shrinkage', str(cfg_dict.get('evidence_shrinkage', 5.0)),
        '--graph-cache-dir', str(cfg_dict.get('graph_cache_dir', '/kaggle/working/runs/stair5_v2/graph_cache')),
        '--artifact-dir', str(artifact_dir),
    ]

    optional_flags = [
        ('--epochs', 'epochs'), ('--batch-size', 'batch_size'),
        ('--lr', 'lr'), ('--weight-decay', 'weight_decay'),
        ('--lambda-lhc', 'lambda_lhc'), ('--lhc-tau', 'lhc_tau'),
        ('--lhc-kappa', 'lhc_kappa'), ('--radius-cap', 'radius_cap'),
        ('--num-workers', 'num_workers'), ('--resume-from', 'resume_from'),
    ]
    for flag, k_name in optional_flags:
        if k_name in cfg_dict and cfg_dict[k_name] is not None:
            cmd.extend([flag, str(cfg_dict[k_name])])

    print(f"🚀 [Khởi chạy lệnh] {' '.join(cmd)}")

    sub_env = os.environ.copy()
    runner_dir = os.path.dirname(os.path.abspath(runner_py))
    sub_env['PYTHONPATH'] = f"{runner_dir}:{sub_env.get('PYTHONPATH', '')}"
    sub_env['PYTHONWARNINGS'] = 'ignore'
    sub_env['PYTHONUNBUFFERED'] = '1'

    with open(log_path, 'w', encoding='utf-8') as f:
        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True, env=sub_env, cwd=runner_dir
        )
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            f.write(line)
            f.flush()
        proc.wait()

    stop_evt.set()
    th.join(timeout=3.0)
    elapsed = time.time() - t0

    print('=' * 80)
    if proc.returncode != 0:
        raise RuntimeError(f'❌ [HUẤN LUYỆN THẤT BẠI] Subprocess kết thúc với mã lỗi {proc.returncode}! Vui lòng kiểm tra file nhật ký: {log_path}')

    print(f'✅ [HOÀN THÀNH HUẤN LUYỆN] Huấn luyện {key.upper()} thành công trong {elapsed/60:.2f} phút ({elapsed:.1f}s)!')

    best_val_ep, val_metrics = extract_best_validation(log_path)
    best_test_ep, test_metrics = extract_test_metrics(log_path)

    print(f'  * Checkpoint Validation Tốt Nhất: Epoch {best_val_ep} (NDCG@20 = {val_metrics.get("NDCG@20", 0.0):.4f})')
    if test_metrics:
        print(f'  * Kết Quả TEST Chính Thức (@Epoch {best_test_ep}):')
        for m, val in test_metrics.items():
            ref_bl = BASELINE_REF.get(key, {}).get(m, 0.0)
            gain_bl = ((val - ref_bl) / ref_bl * 100) if ref_bl > 0 else 0.0
            sign_bl = '+' if gain_bl >= 0 else ''
            print(f'      - {m:12s}: {val:.4f} (vs Baseline: {sign_bl}{gain_bl:.2f}%)')
    else:
        print('  * Kết Quả TEST: Chưa có khối TEST hoàn tất (Pending/Evaluating).')
    print('=' * 80)

# ==============================================================================
# VISUALIZATION UTILITIES (PAPER STANDARD)
# ==============================================================================
def plot_single_dataset_vram(key, dataset_name=None, artifact_dir=None, output_filename=None):
    '''Generates publication-grade model tensor VRAM profile from measured telemetry.'''
    import matplotlib.pyplot as plt

    info = DATASET_PROFILES.get(key, {'name': key.capitalize(), 'color': '#ff7f0e', 'approx_mins': 30.0})
    disp_name = dataset_name if dataset_name else info['name']
    color = info.get('color', '#ff7f0e')

    cfg_item = STAIR5_V2_CONFIGS.get(key, {})
    art_dir = artifact_dir or cfg_item.get('artifact_dir', '')
    telemetry = parse_telemetry_jsonl(art_dir)
    rec_vram = vram_profile.get(key, [])

    fig, ax = plt.subplots(figsize=(10, 4.8), dpi=150)

    if len(rec_vram) > 0:
        steps = np.arange(1, len(rec_vram) + 1)
        ax.plot(steps, rec_vram, color=color, linewidth=2.0, label=f'{disp_name} Host GPU Memory (NVML)', zorder=4)
        ax.fill_between(steps, rec_vram, color=color, alpha=0.15, zorder=3)
        peak_used = max(rec_vram)
        ax.axhline(peak_used, color='#111111', linestyle='--', linewidth=1.3,
                   label=f'Observed Peak Host: {peak_used:.1f} MiB ({peak_used/1024:.2f} GiB)', zorder=5)
        ax.set_xlabel('Monitoring Step (Interval = 2s)', fontsize=10.5, labelpad=8)
        ax.set_xlim(1, max(len(rec_vram), 2))
        ax.set_ylim(0, peak_used * 1.25)
    elif telemetry and any(r.get('peak_allocated_gib', 0) > 0 for r in telemetry):
        epochs = [r['epoch'] for r in telemetry]
        peaks_mib = [r['peak_allocated_gib'] * 1024 for r in telemetry]
        ax.plot(epochs, peaks_mib, color=color, linewidth=2.0, label=f'{disp_name} PyTorch Allocated Peak', zorder=4)
        ax.fill_between(epochs, peaks_mib, color=color, alpha=0.15, zorder=3)
        peak_used = max(peaks_mib)
        ax.axhline(peak_used, color='#111111', linestyle='--', linewidth=1.3,
                   label=f'Peak Tensor Allocated: {peak_used:.1f} MiB ({peak_used/1024:.2f} GiB)', zorder=5)
        ax.set_xlabel('Training Epoch', fontsize=10.5, labelpad=8)
        ax.set_xlim(1, max(len(epochs), 2))
        ax.set_ylim(0, peak_used * 1.25 if peak_used > 0 else 100)
    else:
        base_tensor_peak = 892.4 if key == 'sports' else (668.5 if key == 'baby' else 2540.0)
        epochs = np.arange(1, 501)
        vram_vals = [base_tensor_peak * (1.0 - 0.05 * np.exp(-e / 25.0)) for e in epochs]
        ax.plot(epochs, vram_vals, color=color, linewidth=1.8, linestyle=':', label=f'{disp_name} Budget Estimate', zorder=4)
        ax.axhline(base_tensor_peak, color='#777777', linestyle='--', linewidth=1.2,
                   label=f'Estimated Peak Tensor: ~{base_tensor_peak:.1f} MiB ({base_tensor_peak/1024:.2f} GiB)', zorder=5)
        ax.set_xlabel('Training Epoch (Pre-training Preview)', fontsize=10.5, labelpad=8)
        ax.set_xlim(1, 500)
        ax.set_ylim(0, base_tensor_peak * 1.25)

    ax.set_ylabel('Model Tensor Memory (MiB)', fontsize=10.5, labelpad=8)
    ax.set_title(f"Model VRAM Profile — {disp_name} (STAIR5-v2 C-HET)", fontsize=12.5, fontweight='bold', pad=12)
    ax.grid(True, linestyle='--', alpha=0.30, zorder=1)
    ax.legend(loc='lower right', fontsize=9.0, framealpha=0.9)

    plt.tight_layout()
    if not output_filename:
        output_filename = f'/kaggle/working/reports/vram_profile_{key}.png'
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ [Model Tensor VRAM Profile Saved] -> {output_filename}")

def plot_single_dataset_learning_curves(key, dataset_name=None, artifact_dir=None, log_path=None, output_filename=None):
    '''Plots clean, 4-panel publication-standard learning dynamics with disentangled BPR and LHC losses.'''
    import matplotlib.pyplot as plt

    info = DATASET_PROFILES.get(key, {'name': key.capitalize(), 'color': '#ff7f0e'})
    disp_name = dataset_name if dataset_name else info['name']

    cfg_item = STAIR5_V2_CONFIGS.get(key, {})
    log_file = log_path or cfg_item.get('log', f'{key}_stair5_v2.log')
    art_dir = artifact_dir or cfg_item.get('artifact_dir', '')

    telemetry = parse_telemetry_jsonl(art_dir)
    bpr_losses, total_losses = parse_training_losses(log_file, telemetry)
    val_ndcg = parse_valid_metric(log_file, 'NDCG@20')
    val_recall = parse_valid_metric(log_file, 'Recall@20')

    preview_mode = (len(bpr_losses) == 0 and len(total_losses) == 0)

    fig, axes = plt.subplots(1, 4, figsize=(22, 4.8), dpi=150)
    title_suffix = ' [Chờ Chạy Huấn Luyện (Pending Run)]' if preview_mode else ''
    fig.suptitle(f'Training Dynamics & Convergence — {disp_name} (STAIR5-v2 C-HET){title_suffix}',
                 fontsize=14, fontweight='bold', y=0.98)

    # 1. BPR Loss vs Total Loss
    ax_loss = axes[0]
    if not preview_mode and (bpr_losses or total_losses):
        if bpr_losses:
            eps_b, l_bpr = zip(*bpr_losses)
            ax_loss.plot(eps_b, l_bpr, color='#1f77b4', lw=1.8, label='Pure BPR Loss (L_bpr)')
        if total_losses:
            eps_t, l_tot = zip(*total_losses)
            ax_loss.plot(eps_t, l_tot, color='#aec7e8', lw=1.4, linestyle='--', label='Total Loss (BPR + λ·LHC)')
        max_ep = max(eps_t[-1] if total_losses else (eps_b[-1] if bpr_losses else 2), 2)
        ax_loss.set_xlim(1, max_ep)
        ax_loss.legend(loc='upper right', fontsize=8.0)
    else:
        ax_loss.text(0.5, 0.5, "Chưa có dữ liệu huấn luyện\n(Thực thi cell huấn luyện để ghi nhận loss)",
                     ha='center', va='center', transform=ax_loss.transAxes, color='#777777', fontsize=10.5)
    ax_loss.set_title('(a) Training Loss Trajectory', fontweight='bold', fontsize=11.5)
    ax_loss.set_xlabel('Training Epoch', fontsize=10)
    ax_loss.set_ylabel('Loss Value', fontsize=10)
    ax_loss.grid(True, linestyle='--', alpha=0.35)

    # 2. Validation NDCG@20
    ax_ndcg = axes[1]
    best_n_ep, best_n_val = 0, 0.0
    if not preview_mode and val_ndcg:
        eps_n, ndcgs = zip(*val_ndcg)
        best_n_ep, best_n_val = max(val_ndcg, key=lambda x: x[1])
        ax_ndcg.plot(eps_n, ndcgs, color='#2ca02c', lw=2.0, label='Validation NDCG@20')
        ax_ndcg.axvline(best_n_ep, color='#777777', linestyle='--', lw=1.2, alpha=0.75, label=f'Best Epoch ({best_n_ep})')
        ax_ndcg.legend(loc='lower right', fontsize=8.5)
    else:
        ax_ndcg.text(0.5, 0.5, "Chưa có dữ liệu validation NDCG@20\n(Đang chờ huấn luyện)",
                     ha='center', va='center', transform=ax_ndcg.transAxes, color='#777777', fontsize=10.5)
    ax_ndcg.set_title('(b) Validation NDCG@20 Progression', fontweight='bold', fontsize=11.5)
    ax_ndcg.set_xlabel('Validation Epoch', fontsize=10)
    ax_ndcg.set_ylabel('NDCG@20 Score', fontsize=10)
    ax_ndcg.grid(True, linestyle='--', alpha=0.35)

    # 3. Validation Recall@20
    ax_rec = axes[2]
    best_r_ep, best_r_val = 0, 0.0
    if not preview_mode and val_recall:
        eps_r, recalls = zip(*val_recall)
        best_r_ep, best_r_val = max(val_recall, key=lambda x: x[1])
        ax_rec.plot(eps_r, recalls, color='#9467bd', lw=2.0, label='Validation Recall@20')
        ax_rec.axvline(best_r_ep, color='#777777', linestyle='--', lw=1.2, alpha=0.75, label=f'Best Epoch ({best_r_ep})')
        ax_rec.legend(loc='lower right', fontsize=8.5)
    else:
        ax_rec.text(0.5, 0.5, "Chưa có dữ liệu validation Recall@20\n(Đang chờ huấn luyện)",
                     ha='center', va='center', transform=ax_rec.transAxes, color='#777777', fontsize=10.5)
    ax_rec.set_title('(c) Validation Recall@20 Progression', fontweight='bold', fontsize=11.5)
    ax_rec.set_xlabel('Validation Epoch', fontsize=10)
    ax_rec.set_ylabel('Recall@20 Score', fontsize=10)
    ax_rec.grid(True, linestyle='--', alpha=0.35)

    # 4. Telemetry / Operational Dynamics
    ax_dyn = axes[3]
    has_lhc = telemetry and any(r.get('cl_loss', 0) > 0 for r in telemetry)
    if not preview_mode and telemetry:
        eps = [r['epoch'] for r in telemetry]
        if has_lhc:
            cl = [r.get('cl_loss', 0) for r in telemetry]
            lams = [r.get('lambda', 0) for r in telemetry]
            ax_dyn.plot(eps, cl, color='#d62728', lw=1.8, label='LHC Contrastive Loss')
            ax_lam = ax_dyn.twinx()
            ax_lam.plot(eps, lams, color='#17becf', linestyle='--', lw=2.0, label='Warmup λ(t)')
            ax_lam.set_ylabel('λ(t) Weight', color='#17becf', fontsize=10)
            lines1, labels1 = ax_dyn.get_legend_handles_labels()
            lines2, labels2 = ax_lam.get_legend_handles_labels()
            ax_dyn.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8.0)
            ax_dyn.set_title('(d) LHC Contrastive Loss & λ(t)', fontweight='bold', fontsize=11.5)
            ax_dyn.set_ylabel('LHC Loss', fontsize=10)
        else:
            sps = [r.get('samples_per_second', 0) for r in telemetry]
            vram = [r.get('peak_allocated_gib', 0) * 1024 for r in telemetry]
            ax_dyn.plot(eps, sps, color='#ff7f0e', lw=1.8, label='Throughput (samples/s)')
            ax_dyn.set_ylabel('Samples / sec', color='#ff7f0e', fontsize=10)
            ax_vr = ax_dyn.twinx()
            ax_vr.plot(eps, vram, color='#2ca02c', linestyle=':', lw=1.8, label='Peak VRAM (MiB)')
            ax_vr.set_ylabel('Allocated VRAM (MiB)', color='#2ca02c', fontsize=10)
            lines1, labels1 = ax_dyn.get_legend_handles_labels()
            lines2, labels2 = ax_vr.get_legend_handles_labels()
            ax_dyn.legend(lines1 + lines2, lines2 + labels2, loc='lower right', fontsize=8.0)
            ax_dyn.set_title('(d) Throughput & VRAM Telemetry', fontweight='bold', fontsize=11.5)
    else:
        ax_dyn.text(0.5, 0.5, "Chưa có dữ liệu Telemetry\n(Ghi nhận tự động từ STAIR5-v2)",
                    ha='center', va='center', transform=ax_dyn.transAxes, color='#777777', fontsize=10.5)
        ax_dyn.set_title('(d) Operational Telemetry (C-HET)', fontweight='bold', fontsize=11.5)
    ax_dyn.set_xlabel('Training Epoch', fontsize=10)
    ax_dyn.grid(True, linestyle='--', alpha=0.35)

    note_text = f'Trạng thái: Đã hoàn tất huấn luyện tại Epoch {best_n_ep} (NDCG@20 = {best_n_val:.4f}).' if not preview_mode and best_n_ep > 0 else 'Trạng thái: Chưa có dữ liệu huấn luyện thực tế — Đang chờ thực thi.'
    fig.text(0.5, 0.015, note_text, ha='center', fontsize=9.5, color='#333333', style='italic')

    plt.tight_layout(rect=[0, 0.045, 1, 0.96])
    if not output_filename:
        output_filename = f'/kaggle/working/reports/learning_curve_{key}.png'
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ [Learning Dynamics Saved] -> {output_filename}")

## Cell 5 📋 Cấu hình Siêu tham số STAIR5-v2 C-HET (Dataset-Adaptive Matrix)
Cấu hình chuẩn hóa dựa trên `docs/giai_doan_5/STAIR5_v2_Implementation.md` và các YAML v2:
- **Amazon Sports:** `configs/Amazon2014Sports_STAIR5_v2.yaml` ($D=64, L=3, B=1024, \text{lr}=10^{-3}, \text{decay}=0.1, \gamma=0.2$, Arm `ET`, $a=0.25, \eta=0.25, t=5$).
- **Amazon Baby:** `configs/Amazon2014Baby_STAIR5_v2.yaml` ($D=64, L=3, B=1024, \text{lr}=10^{-3}, \text{decay}=0.3, \gamma=0.1$, Arm `ET`, $a=0.25, \eta=0.25, t=5$).
- **Amazon Electronics:** `configs/Amazon2014Electronics_STAIR5_v2.yaml` ($D=64, L=3, B=4096, \text{lr}=10^{-3}, \text{decay}=0.1, \gamma=0.2$, Arm `ET`, $a=0.25, \eta=0.25, t=5$).

*Ghi chú: Toàn bộ siêu tham số đều được truyền trực tiếp qua cờ dòng lệnh CLI bảo đảm tính minh bạch, độc lập và có thể tái lập 100%.*

In [ ]:
# Cell 5: Cấu hình Siêu tham số STAIR5-v2 C-HET (Dataset-Adaptive Matrix)
import os

os.makedirs('/kaggle/working/logs/stair5_v2', exist_ok=True)
os.makedirs('/kaggle/working/reports', exist_ok=True)
os.makedirs('/kaggle/working/runs/stair5_v2/graph_cache', exist_ok=True)

STAIR5_V2_CONFIGS = {
    'sports': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Sports_STAIR5_v2.yaml',
        'log':                '/kaggle/working/logs/stair5_v2/sports_stair5_v2.log',
        'artifact_dir':       '/kaggle/working/runs/stair5_v2/sports_ET_seed1',
        'v2_arm':             'ET',
        'edge_strength':      0.25,
        'edge_mix':           0.25,
        'evidence_shrinkage': 5.0,
        'knn_device':         'auto',
        'knn_chunk_size':     1024,
        'lhc_chunk_size':     256,
        'seed':               1,
        'eval_freq':          5,
    },
    'baby': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Baby_STAIR5_v2.yaml',
        'log':                '/kaggle/working/logs/stair5_v2/baby_stair5_v2.log',
        'artifact_dir':       '/kaggle/working/runs/stair5_v2/baby_ET_seed1',
        'v2_arm':             'ET',
        'edge_strength':      0.25,
        'edge_mix':           0.25,
        'evidence_shrinkage': 5.0,
        'knn_device':         'auto',
        'knn_chunk_size':     1024,
        'lhc_chunk_size':     256,
        'seed':               1,
        'eval_freq':          5,
    },
    'electronics': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Electronics_STAIR5_v2.yaml',
        'log':                '/kaggle/working/logs/stair5_v2/electronics_stair5_v2.log',
        'artifact_dir':       '/kaggle/working/runs/stair5_v2/electronics_ET_seed1',
        'v2_arm':             'ET',
        'edge_strength':      0.25,
        'edge_mix':           0.25,
        'evidence_shrinkage': 5.0,
        'knn_device':         'auto',
        'knn_chunk_size':     1024,
        'lhc_chunk_size':     256,
        'seed':               1,
        'eval_freq':          5,
    },
}

print('✅ Ma trận cấu hình STAIR5-v2 C-HET đã nạp thành công:')
for k, v in STAIR5_V2_CONFIGS.items():
    print(f"  * {k.upper():12s} -> Arm: {v['v2_arm']} | Config: {os.path.basename(v['yaml'])} | Log: {os.path.basename(v['log'])}")

## Cell 6a 🏋️ Huấn luyện Pha A — Amazon Sports (Pilot & Đề Xuất Chính C-HET: ET Arm)
Chạy thực nghiệm huấn luyện kiến trúc đề xuất chính **STAIR5-v2 (ET)** trên tập **Amazon Sports** (35,598 users, 18,357 items, 296K interactions).  
- **Cấu hình:** $a=0.25, \eta=0.25, t=5.0$, Arm `ET` (tắt LHC, không tiêu tốn tính toán phụ).  
- **Đánh giá:** Lựa chọn checkpoint theo chuẩn `validation NDCG@20` cao nhất.

In [ ]:
# Cell 6a: Training STAIR5-v2 C-HET on Amazon Sports (Pha A)
DATA_ROOT = '/kaggle/data'

if 'sports' in prepared_data:
    cfg_s = STAIR5_V2_CONFIGS['sports']
    run_training_stair5_v2(
        key          = 'sports',
        config_yaml  = cfg_s['yaml'],
        data_root    = DATA_ROOT,
        log_path     = cfg_s['log'],
        artifact_dir = cfg_s['artifact_dir'],
    )
else:
    print("⚠️ Bỏ qua Amazon Sports do chưa chuẩn bị xong dữ liệu.")

## Cell 6b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Sports (Paper Standard)
Trực quan hóa mức tiêu thụ VRAM thuần bộ nhớ tensor trong suốt quá trình huấn luyện Amazon Sports.

In [ ]:
# Cell 6b: Model Tensor VRAM Profile — Amazon Sports (Paper Standard)
plot_single_dataset_vram(
    key             = 'sports',
    dataset_name    = 'Amazon Sports',
    artifact_dir    = STAIR5_V2_CONFIGS['sports']['artifact_dir'],
    output_filename = '/kaggle/working/reports/vram_profile_sports.png'
)

## Cell 6c 📈 Động Lực Học & Quá Trình Hội Tụ (Learning Dynamics) — Amazon Sports (Paper Standard)
Trực quan hóa tiến trình huấn luyện STAIR5-v2 trên tập **Amazon Sports**: Phân tách Pure BPR Loss và Total Loss, Validation NDCG@20, Validation Recall@20, cùng thông lượng mẫu và VRAM telemetry.

In [ ]:
# Cell 6c: Learning Dynamics & Convergence Profiles — Amazon Sports (Paper Standard)
plot_single_dataset_learning_curves(
    key             = 'sports',
    dataset_name    = 'Amazon Sports',
    artifact_dir    = STAIR5_V2_CONFIGS['sports']['artifact_dir'],
    log_path        = STAIR5_V2_CONFIGS['sports']['log'],
    output_filename = '/kaggle/working/reports/learning_curve_sports.png'
)

## Cell 7a 🏋️ Huấn luyện Pha B — Amazon Baby (Kiểm chứng An toàn Over-smoothing)
Chạy thực nghiệm huấn luyện trên tập **Amazon Baby** (19,445 users, 7,050 items, 160K interactions).  
- **Cấu hình:** Kế thừa đúng $\gamma=0.1$, weight decay $0.3$, $a=0.25, \eta=0.25, t=5.0$, Arm `ET`.  
- **Mục tiêu:** Kiểm chứng hiệu chỉnh cạnh tin cậy có khắc phục hiện tượng suy giảm từng thấy ở v1 trên Baby hay không.

In [ ]:
# Cell 7a: Training STAIR5-v2 C-HET on Amazon Baby (Pha B)
DATA_ROOT = '/kaggle/data'

if 'baby' in prepared_data:
    cfg_b = STAIR5_V2_CONFIGS['baby']
    run_training_stair5_v2(
        key          = 'baby',
        config_yaml  = cfg_b['yaml'],
        data_root    = DATA_ROOT,
        log_path     = cfg_b['log'],
        artifact_dir = cfg_b['artifact_dir'],
    )
else:
    print("⚠️ Bỏ qua Amazon Baby do chưa chuẩn bị xong dữ liệu.")

## Cell 7b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Baby (Paper Standard)
Trực quan hóa mức tiêu thụ VRAM thuần bộ nhớ tensor trong suốt quá trình huấn luyện Amazon Baby.

In [ ]:
# Cell 7b: Model Tensor VRAM Profile — Amazon Baby (Paper Standard)
plot_single_dataset_vram(
    key             = 'baby',
    dataset_name    = 'Amazon Baby',
    artifact_dir    = STAIR5_V2_CONFIGS['baby']['artifact_dir'],
    output_filename = '/kaggle/working/reports/vram_profile_baby.png'
)

## Cell 7c 📈 Động Lực Học & Quá Trình Hội Tụ (Learning Dynamics) — Amazon Baby (Paper Standard)
Trực quan hóa tiến trình huấn luyện STAIR5-v2 trên tập **Amazon Baby**.

In [ ]:
# Cell 7c: Learning Dynamics & Convergence Profiles — Amazon Baby (Paper Standard)
plot_single_dataset_learning_curves(
    key             = 'baby',
    dataset_name    = 'Amazon Baby',
    artifact_dir    = STAIR5_V2_CONFIGS['baby']['artifact_dir'],
    log_path        = STAIR5_V2_CONFIGS['baby']['log'],
    output_filename = '/kaggle/working/reports/learning_curve_baby.png'
)

## Cell 8a 🏋️ Huấn luyện Pha C — Amazon Electronics (~1.7M Tương tác, Quy mô Khổng lồ)
Chạy thực nghiệm kiểm chứng khả năng mở rộng quy mô trên tập **Amazon Electronics** (192,403 users, 63,001 items, 1.69M interactions).  
- **Cấu hình:** Batch size 4096, $\gamma=0.2$, decay $0.1$, Arm `ET`, $a=0.25, \eta=0.25, t=5.0$.  
- **Tối ưu tính toán:** Bounded top-k chunking (1024), candidate-restricted common counts, ma trận thưa CSR BSC Smoother, không tốn VRAM cho auxiliary branch.

In [ ]:
# Cell 8a: Training STAIR5-v2 C-HET on Amazon Electronics (Pha C)
DATA_ROOT = '/kaggle/data'

if 'electronics' in prepared_data:
    cfg_e = STAIR5_V2_CONFIGS['electronics']
    run_training_stair5_v2(
        key          = 'electronics',
        config_yaml  = cfg_e['yaml'],
        data_root    = DATA_ROOT,
        log_path     = cfg_e['log'],
        artifact_dir = cfg_e['artifact_dir'],
    )
else:
    print("⚠️ Bỏ qua Amazon Electronics do chưa chuẩn bị xong dữ liệu.")

## Cell 8b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Electronics (Paper Standard)
Trực quan hóa mức tiêu thụ VRAM thuần bộ nhớ tensor trong suốt quá trình huấn luyện Amazon Electronics.

In [ ]:
# Cell 8b: Model Tensor VRAM Profile — Amazon Electronics (Paper Standard)
plot_single_dataset_vram(
    key             = 'electronics',
    dataset_name    = 'Amazon Electronics',
    artifact_dir    = STAIR5_V2_CONFIGS['electronics']['artifact_dir'],
    output_filename = '/kaggle/working/reports/vram_profile_electronics.png'
)

## Cell 8c 📈 Động Lực Học & Quá Trình Hội Tụ (Learning Dynamics) — Amazon Electronics (Paper Standard)
Trực quan hóa tiến trình huấn luyện STAIR5-v2 trên tập **Amazon Electronics**.

In [ ]:
# Cell 8c: Learning Dynamics & Convergence Profiles — Amazon Electronics (Paper Standard)
plot_single_dataset_learning_curves(
    key             = 'electronics',
    dataset_name    = 'Amazon Electronics',
    artifact_dir    = STAIR5_V2_CONFIGS['electronics']['artifact_dir'],
    log_path        = STAIR5_V2_CONFIGS['electronics']['log'],
    output_filename = '/kaggle/working/reports/learning_curve_electronics.png'
)

## Cell 9 📊 Bảng So sánh Tổng hợp Ablation Study Đa Phiên bản (3 Datasets — Đầy đủ 4 Chỉ số)
Trích xuất tự động và đối chiếu toàn diện 4 chỉ số chuẩn: **Recall@10, Recall@20, NDCG@10, NDCG@20** giữa:
1. STAIR Baseline (Chuẩn MMRec / Tái lập)
2. STAIR-BSC-Reweight (v5 Giai đoạn 2)
3. STAIR-MHD v3 (Giai đoạn 4)
4. STAIR-LHC v1 (Giai đoạn 5 v1)
5. **★ STAIR5-v2 / C-HET (Giai đoạn 5 v2: Confidence-calibrated Edge Trust)**

*Nguyên tắc trung thực học thuật:* Chỉ những tập dữ liệu đã chạy hoàn tất và có kết quả kiểm thử chính thức (`TEST @Epoch`) từ việc load checkpoint tốt nhất mới được ghi nhận chỉ số thực nghiệm. Khi chưa hoàn tất run, bảng hiển thị rõ `N/A`, tuyệt đối không tự ý gán nhãn mục tiêu.

In [ ]:
# Cell 9: Bảng so sánh Ablation Study toàn diện (Recall@10, Recall@20, NDCG@10, NDCG@20)
import os
try:
    from prettytable import PrettyTable
    USE_PRETTYTABLE = True
except ImportError:
    USE_PRETTYTABLE = False

BASELINE = {
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V5_RESULTS = {
    'sports':      {'Recall@10': 0.0753, 'Recall@20': 0.1113, 'NDCG@10': 0.0415, 'NDCG@20': 0.0508},
    'baby':        {'Recall@10': 0.0669, 'Recall@20': 0.1027, 'NDCG@10': 0.0362, 'NDCG@20': 0.0454},
    'electronics': {'Recall@10': 0.0451, 'Recall@20': 0.0678, 'NDCG@10': 0.0252, 'NDCG@20': 0.0311},
}

MHD_V3_RESULTS = {
    'sports':      {'Recall@10': 0.0758, 'Recall@20': 0.1122, 'NDCG@10': 0.0418, 'NDCG@20': 0.0512},
    'baby':        {'Recall@10': 0.0678, 'Recall@20': 0.1030, 'NDCG@10': 0.0362, 'NDCG@20': 0.0452},
    'electronics': {'Recall@10': 0.0460, 'Recall@20': 0.0685, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
}

V1_H0_RESULTS = {
    'sports':      {'Recall@10': 0.0747, 'Recall@20': 0.1133, 'NDCG@10': 0.0407, 'NDCG@20': 0.0506},
    'baby':        {'Recall@10': 0.0660, 'Recall@20': 0.1030, 'NDCG@10': 0.0352, 'NDCG@20': 0.0447},
    'electronics': {'Recall@10': 0.0435, 'Recall@20': 0.0666, 'NDCG@10': 0.0241, 'NDCG@20': 0.0301},
}

headers = [
    'Dataset', 'Phiên bản', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20',
    'Δ vs Base R@20 (%)', 'Δ vs Base N@20 (%)', 'Δ vs v1 N@20 (%)', 'Trạng thái Nghiệm thu'
]

rows = []

for key in ['sports', 'baby', 'electronics']:
    bl = BASELINE[key]
    v5 = V5_RESULTS[key]
    v3 = MHD_V3_RESULTS[key]
    v1 = V1_H0_RESULTS[key]
    d_name = key.upper()

    # 1. Baseline
    rows.append([
        d_name, 'STAIR Baseline',
        f"{bl['Recall@10']:.4f}", f"{bl['Recall@20']:.4f}",
        f"{bl['NDCG@10']:.4f}", f"{bl['NDCG@20']:.4f}",
        '0.00%', '0.00%', '-', 'Mốc chuẩn MMRec'
    ])

    # 2. v5 (Giai đoạn 2)
    d_r20_v5 = (v5['Recall@20'] - bl['Recall@20']) / bl['Recall@20'] * 100
    d_n20_v5 = (v5['NDCG@20'] - bl['NDCG@20']) / bl['NDCG@20'] * 100
    rows.append([
        d_name, 'STAIR-BSC-Reweight (v5)',
        f"{v5['Recall@10']:.4f}", f"{v5['Recall@20']:.4f}",
        f"{v5['NDCG@10']:.4f}", f"{v5['NDCG@20']:.4f}",
        f"{d_r20_v5:+.2f}%", f"{d_n20_v5:+.2f}%", '-', 'Giai đoạn 2 (Graph-level)'
    ])

    # 3. MHD v3 (Giai đoạn 4)
    d_r20_v3 = (v3['Recall@20'] - bl['Recall@20']) / bl['Recall@20'] * 100
    d_n20_v3 = (v3['NDCG@20'] - bl['NDCG@20']) / bl['NDCG@20'] * 100
    rows.append([
        d_name, 'STAIR-MHD v3',
        f"{v3['Recall@10']:.4f}", f"{v3['Recall@20']:.4f}",
        f"{v3['NDCG@10']:.4f}", f"{v3['NDCG@20']:.4f}",
        f"{d_r20_v3:+.2f}%", f"{d_n20_v3:+.2f}%", '-', 'Giai đoạn 4 (Hypergraph)'
    ])

    # 4. STAIR-LHC v1 (Giai đoạn 5 v1)
    d_r20_v1 = (v1['Recall@20'] - bl['Recall@20']) / bl['Recall@20'] * 100
    d_n20_v1 = (v1['NDCG@20'] - bl['NDCG@20']) / bl['NDCG@20'] * 100
    rows.append([
        d_name, 'STAIR-LHC v1',
        f"{v1['Recall@10']:.4f}", f"{v1['Recall@20']:.4f}",
        f"{v1['NDCG@10']:.4f}", f"{v1['NDCG@20']:.4f}",
        f"{d_r20_v1:+.2f}%", f"{d_n20_v1:+.2f}%", '0.00%', 'Giai đoạn 5 (v1 Lorentz)'
    ])

    # 5. STAIR5-v2 C-HET (Đo trực tiếp từ log)
    log_file = STAIR5_V2_CONFIGS[key]['log']
    best_ep, test_m = extract_test_metrics(log_file)

    if test_m and len(test_m) >= 4:
        d_r20_v2 = (test_m['Recall@20'] - bl['Recall@20']) / bl['Recall@20'] * 100
        d_n20_v2 = (test_m['NDCG@20'] - bl['NDCG@20']) / bl['NDCG@20'] * 100
        d_n20_vs_v1 = (test_m['NDCG@20'] - v1['NDCG@20']) / v1['NDCG@20'] * 100
        note = f'★ Verified TEST @Ep {best_ep}'
        rows.append([
            d_name, '★ STAIR5-v2 (C-HET ET)',
            f"{test_m['Recall@10']:.4f}", f"{test_m['Recall@20']:.4f}",
            f"{test_m['NDCG@10']:.4f}", f"{test_m['NDCG@20']:.4f}",
            f"{d_r20_v2:+.2f}%", f"{d_n20_v2:+.2f}%", f"{d_n20_vs_v1:+.2f}%", note
        ])
    else:
        rows.append([
            d_name, '★ STAIR5-v2 (C-HET ET)',
            'N/A', 'N/A', 'N/A', 'N/A',
            'N/A', 'N/A', 'N/A', 'Chưa có kết quả TEST (N/A)'
        ])

print('=' * 115)
print('BẢNG TỔNG HỢP SO SÁNH ABLATION STUDY ĐA PHIÊN BẢN (STAIR BASELINE vs v5 vs MHD v3 vs v1 vs STAIR5-v2):')
print('=' * 115)

use_pt = globals().get('USE_PRETTYTABLE', False)
if use_pt:
    try:
        t = PrettyTable()
        t.field_names = headers
        for r in rows:
            t.add_row(r)
        print(t)
    except Exception:
        use_pt = False

if not use_pt:
    print(' | '.join(headers))
    print('-' * 115)
    for r in rows:
        print(f"{r[0]:12s} | {r[1]:26s} | {r[2]:6s} | {r[3]:6s} | {r[4]:6s} | {r[5]:6s} | {r[6]:12s} | {r[7]:12s} | {r[8]:10s} | {r[9]}")

## Cell 10 📦 Đóng gói Lưu trữ Toàn bộ Minh chứng, Logs, Manifests & Checkpoints
- Tự động nén toàn bộ artifacts: `manifest.json`, `training_telemetry.jsonl`, `best_model.pt`, `training_checkpoint.pt`, các biểu đồ PNG và file logs.
- Tạo mã băm SHA-256 đối soát tính toàn vẹn dữ liệu (`checksums.json`).
- Cho phép tải về trực tiếp từ Kaggle Working Output.

In [ ]:
# Cell 10: Đóng gói Lưu trữ Toàn bộ Minh chứng (Logs, Checkpoints, Manifests, Plots)
import os, zipfile, hashlib, json
from pathlib import Path

WORKING_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPORT_DIR = WORKING_DIR / 'reports'
RUN_DIR = WORKING_DIR / 'runs' / 'stair5_v2'
LOG_DIR = WORKING_DIR / 'logs' / 'stair5_v2'
ZIP_PATH = WORKING_DIR / 'stair5_v2_artifacts.zip'

def sha256_file(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

collected_files = []
for directory in [REPORT_DIR, RUN_DIR, LOG_DIR]:
    if directory.exists():
        for root, dirs, files in os.walk(directory):
            for file in files:
                full_path = os.path.join(root, file)
                # Loại trừ file tạm hoặc symlink data lớn
                if not os.path.islink(full_path):
                    collected_files.append(full_path)

checksums = {}
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for file_path in collected_files:
        rel_path = os.path.relpath(file_path, WORKING_DIR)
        zipf.write(file_path, arcname=rel_path)
        checksums[rel_path] = sha256_file(file_path)

checksum_path = WORKING_DIR / 'checksums_stair5_v2.json'
with open(checksum_path, 'w', encoding='utf-8') as f:
    json.dump(checksums, f, indent=2)

print('=' * 80)
print(f'📦 ĐÓNG GÓI ARTIFACTS HOÀN TẤT:')
print(f'  * File nén          : {ZIP_PATH}')
print(f'  * Dung lượng        : {os.path.getsize(ZIP_PATH) / (1024**2):.2f} MB')
print(f'  * Số lượng file     : {len(collected_files)}')
print(f'  * Checksums SHA-256 : {checksum_path}')
print('=' * 80)